In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.9 The Grand Canonical Ensemble, Fluctuations, and the Equivalence of Ensembles

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.9",
    title="The Grand Canonical Ensemble, Fluctuations, and the Equivalence of Ensembles",
    blurb="The ensembles converge. We open the system to particle exchange, "
    "discover that its number fluctuations are the compressibility just as its "
    "energy fluctuations were the heat capacity, and find that every response "
    "function is a fluctuation. Because those fluctuations shrink as 1/√N, the "
    "three ensembles tell the same story — until a critical point, where they "
    "do not.",
    difficulty="advanced",
    estimate="200–240 min",
)

## Notebook overview

We now hold the whole machinery of equilibrium statistical mechanics, and this notebook is where
it closes into a single structure. We have met two ensembles: the **microcanonical** ([§5.4](microstates-entropy-temperature.ipynb)), an
isolated system at fixed energy and particle number, and the **canonical** ([§5.8](partition-function.ipynb)), a system at
fixed temperature exchanging energy with a heat bath. There is one more degree of freedom to set
free — the particle number — and doing so gives the **grand canonical** ensemble: a system at
fixed temperature *and* chemical potential, exchanging both energy and particles with a reservoir.
It completes the family, and it is the natural language for any open system, from a gas in a
subvolume to adsorption to chemical equilibrium.

But the real subject of this notebook is **fluctuations**, and they are its spine rather than a
subsection. Two ideas, developed in full, organize everything. The first is the **fluctuation–response
principle**: the variance of an extensive quantity *equals* the response of that quantity
to its conjugate field. The canonical energy fluctuation was the heat capacity ([§5.8](partition-function.ipynb)),
$\operatorname{Var}(E)=kT^2C_V$; here the grand-canonical particle-number fluctuation will turn out
to be the compressibility, $\operatorname{Var}(N)=kT(\partial\langle N\rangle/\partial\mu)$; and
the magnetization fluctuation is the susceptibility, $\operatorname{Var}(M)=kT\chi$. These are not
three coincidences but one theorem — the static fluctuation–dissipation theorem — and it is how
simulations actually measure response functions, by watching a system jitter at equilibrium.

The second idea is that fluctuations are what make the three ensembles **equivalent**. A canonical
system does not have a fixed energy, and a grand-canonical system does not have a fixed particle
number — but their distributions are so sharply peaked (relative width $\sim1/\sqrt N$, the result
of [§5.3](large-n-limit.ipynb)) that at macroscopic $N$ the energy and number are pinned to their averages just as firmly
as if they were held fixed. So the microcanonical, canonical, and grand canonical descriptions
give identical thermodynamics in the limit, and one is free to choose whichever is most convenient.
The open ideal gas makes the arsenal vivid: its particle number is **Poisson** distributed (the
[§5.2](probability.ipynb) payoff), with $\langle N\rangle=\operatorname{Var}(N)$ and $\sigma_N/\langle N\rangle=
1/\sqrt{\langle N\rangle}$, computed in the log space of [§5.3](large-n-limit.ipynb) and [§0.1](../00-foundations/floating-point.ipynb).

The notebook runs long by design, and it ends by finding the one crack in the equivalence. Where a
response function *diverges*, the fluctuation–response relations say the fluctuations become
macroscopic — they do *not* vanish as $1/\sqrt N$ — and the ensembles can disagree. That place is
a **critical point**, and far from being a pathology it is where the most interesting physics
lives. It is exactly where the Ising model of [§5.10](ising-emergence-universality.ipynb) takes us: a diverging susceptibility,
system-spanning fluctuations, and the spontaneous emergence of order.

> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the grand-canonical probabilities normalized by $\Xi$; the open ideal gas's **Poisson**
> number with $\langle N\rangle=\operatorname{Var}(N)=zZ_1$; $\operatorname{Var}(N)=kT(\partial
> \langle N\rangle/\partial\mu)$ in two forms; the energy fluctuation as $C_V$; the canonical
> energy distribution peaking at the microcanonical energy with width $\propto1/\sqrt N$; the
> Poisson collapse $\sigma_N/\langle N\rangle=1/\sqrt{\langle N\rangle}$; one quantity computed
> three ways agreeing; a response function read off equilibrium fluctuations; and the divergence
> that signals a critical point. A ✓ is strong evidence; a ✗ is a prompt to *locate the
> discrepancy*.
>
> **Scope.** The grand canonical ensemble, the fluctuation–response relations, and the equivalence
> of ensembles, classically. The critical point where equivalence breaks is set up here and
> quantified in [§5.10](ising-emergence-universality.ipynb); the quantum grand canonical ensemble (Fermi–Dirac, Bose–Einstein) is Volume
> VII. See Pathria & Beale; Kardar; Landau & Lifshitz, *Statistical Physics*; and Notebooks [§5.2](probability.ipynb)
> (Poisson), [§5.3](large-n-limit.ipynb) ($1/\sqrt N$), [§5.7](potentials-legendre-maxwell.ipynb) ($\Phi$, response functions), [§5.8](partition-function.ipynb) ($\operatorname{Var}(E)=
> kT^2C_V$).

## Theory in brief

### The grand canonical ensemble

Let a system exchange both energy and particles with a reservoir at temperature $T$ and chemical
potential $\mu$. The probability of a microstate $s$ with energy $E_s$ and particle number $N_s$
is

```{math}
:label: eq-grand
P_s=\frac{e^{-\beta(E_s-\mu N_s)}}{\Xi},\qquad
\Xi=\sum_s e^{-\beta(E_s-\mu N_s)}=\sum_N e^{\beta\mu N}Z_N ,
```

where $\Xi$ is the **grand partition function**, $z=e^{\beta\mu}$ the **fugacity**, and $Z_N$ the
canonical partition function at fixed $N$. The **grand potential** is $\Phi=-kT\ln\Xi$, with
$d\Phi=-S\,dT-P\,dV-N\,d\mu$ — the potential [§5.7](potentials-legendre-maxwell.ipynb) built by Legendre-transforming in $N\to\mu$, now
grounded statistically (as the canonical ensemble grounded $F$). The distribution {eq}`eq-grand`
comes from the reservoir argument of [§5.4](microstates-entropy-temperature.ipynb), expanded now in
two small quantities (Exercise 1 sketches it; Pathria & Beale, Ch. 4, carry it out in full).

### The open ideal gas: Poisson particle number

For the ideal gas $Z_N=Z_1^N/N!$, so the sum is an exponential series,

```{math}
:label: eq-gc-poisson
\Xi=\sum_N\frac{(zZ_1)^N}{N!}=e^{zZ_1},\qquad
P(N)=\frac{(zZ_1)^N e^{-zZ_1}}{N!},
```

a **Poisson** distribution (the [§5.2](probability.ipynb) distribution, now the equilibrium statistics of an open gas)
with mean $\langle N\rangle=zZ_1$. Its signature is $\langle N\rangle=\operatorname{Var}(N)$, hence
$\sigma_N/\langle N\rangle=1/\sqrt{\langle N\rangle}$. We evaluate the pmf in log space with
`scipy.special.gammaln` (the [§5.3](large-n-limit.ipynb)/[§0.1](../00-foundations/floating-point.ipynb) lesson).

### Fluctuation–response relations

The variance of an extensive quantity is the response of that quantity to its conjugate field —
the **static fluctuation–dissipation theorem**,

```{math}
:label: eq-fluct-response
\operatorname{Var}(E)=kT^2C_V,\qquad
\operatorname{Var}(N)=kT\Big(\frac{\partial\langle N\rangle}{\partial\mu}\Big)_{T,V}=\frac{\langle N\rangle^2kT\kappa_T}{V},\qquad
\operatorname{Var}(M)=kT\chi .
```

Energy fluctuations are the heat capacity ([§5.8](partition-function.ipynb)); particle-number fluctuations are the isothermal
compressibility $\kappa_T=-\tfrac1V(\partial V/\partial P)_T$; magnetization fluctuations are the
susceptibility. A susceptibility — the derivative of an extensive variable with respect to its
conjugate intensive field — *equals* the variance of that variable, up to $kT$ ($kT^2$ for the
energy–heat-capacity pair). Response **is**
fluctuation. It lets one read a thermodynamic response straight off a system's equilibrium jitter.

### The equivalence of the three ensembles

The microcanonical (fixed $E,N$), canonical (fixed $T$), and grand canonical (fixed $T,\mu$)
ensembles give the *same* thermodynamics in the thermodynamic limit, and the reason is
fluctuations,

```{math}
:label: eq-gc-equivalence
\frac{\sigma_E}{\langle E\rangle}\sim\frac{1}{\sqrt N},\qquad \frac{\sigma_N}{\langle N\rangle}\sim\frac{1}{\sqrt{\langle N\rangle}} .
```

The canonical energy distribution $P(E)\propto\Omega(E)e^{-\beta E}$ is sharply peaked at the
energy where $\partial S/\partial E=1/T$ — the *microcanonical* energy at that temperature — with
relative width $\propto1/\sqrt N$. As $N\to\infty$ the canonical ensemble effectively fixes the
energy; likewise the grand-canonical Poisson number collapses onto the canonical $N$. The
ensembles are interchangeable because their distinguishing fluctuations are negligible ([§5.3](large-n-limit.ipynb)).

### Where equivalence breaks: critical points

By the fluctuation–response relations, a **diverging** response ($C_V$, $\kappa_T$, or $\chi$) is a
**diverging** fluctuation,

```{math}
:label: eq-criticality
\chi\to\infty \;\Longleftrightarrow\; \operatorname{Var}(M)\to\infty\quad(\text{not}\sim1/\sqrt N),
```

so at a **critical point** the fluctuations become macroscopic, span the system, and ensemble
equivalence can fail. This is not a flaw to avoid; it is where phase transitions, critical
phenomena, and universality live — the Ising critical point of [§5.10](ising-emergence-universality.ipynb).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from matplotlib.patches import FancyArrowPatch, Rectangle
from scipy.special import gammaln

from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Units throughout: k = 1; energies and temperatures are pure numbers (ε = 1 for the two-state gap).


def poisson_pmf_logspace(N, lam):
    """The Poisson pmf P(N) = λ^N e^(−λ)/N!, evaluated in log space (eq-poisson).

    Computes ln P = N ln λ − λ − ln N! with ``scipy.special.gammaln`` (which gives
    ln Γ, so ln N! = ln Γ(N+1)) and exponentiates — the §5.3/§0.1 lesson, since the
    factorial in the denominator overflows for the large ⟨N⟩ of a macroscopic gas.

    Parameters
    ----------
    N : int or numpy.ndarray
        Particle number(s) at which to evaluate the pmf.
    lam : float
        The Poisson mean λ = ⟨N⟩ = z·Z1.

    Returns
    -------
    float or numpy.ndarray
        The probability P(N).
    """
    N = np.asarray(N, dtype=float)
    return np.exp(N * np.log(lam) - lam - gammaln(N + 1.0))


def canonical_energy_distribution(N, beta, eps=1.0):
    """The canonical energy distribution P(E) ∝ Ω(E) e^(−βE) of N two-state spins.

    Each spin has energy ±ε; with n_up in the +ε state the total energy is
    E = (2·n_up − N)ε and the multiplicity is Ω = C(N, n_up). The distribution is
    built in log space (``scipy.special.gammaln`` for ln C(N, n_up)) and normalized.
    It is sharply peaked at the microcanonical energy for the temperature 1/β, with
    relative width ∝ 1/√N.

    Parameters
    ----------
    N : int
        Number of spins.
    beta : float
        Inverse temperature 1/kT.
    eps : float, optional
        Half the level splitting (default ``1.0``).

    Returns
    -------
    tuple of numpy.ndarray
        ``(E, P)`` — the energy values and their probabilities.
    """
    n_up = np.arange(0, N + 1)
    E = (2 * n_up - N) * eps
    ln_omega = gammaln(N + 1.0) - gammaln(n_up + 1.0) - gammaln(N - n_up + 1.0)
    ln_w = ln_omega - beta * E
    w = np.exp(ln_w - ln_w.max())
    return E, w / w.sum()


def grand_canonical_ideal_gas(z, Z1):
    """Grand-canonical statistics of the open ideal gas: Ξ = e^(z·Z1) with Poisson number (eq-poisson).

    Returns the mean and variance of the particle number, both equal to z·Z1 (the Poisson
    signature ⟨N⟩ = Var(N)), summed over the log-space Poisson pmf.

    Parameters
    ----------
    z : float
        Fugacity z = e^(βμ).
    Z1 : float
        Single-particle canonical partition function.

    Returns
    -------
    dict
        ``{"lam", "mean_N", "var_N"}`` — the Poisson mean and the sampled mean/variance.
    """
    lam = z * Z1
    N = np.arange(0, int(lam + 12 * np.sqrt(lam)) + 1)
    P = poisson_pmf_logspace(N, lam)
    mean_N = np.sum(N * P)
    var_N = np.sum((N - mean_N) ** 2 * P)
    return {"lam": lam, "mean_N": mean_N, "var_N": var_N}

## Exercise 1 — The grand canonical ensemble

We complete the family of ensembles by letting one more thing vary. The microcanonical ensemble
fixed energy and number; the canonical ensemble ([§5.8](partition-function.ipynb)) freed the energy, putting the system in
contact with a heat bath at temperature $T$. Now we free the **particle number** as well: the
system sits in contact with a reservoir that supplies and absorbs both energy and particles, at
fixed temperature $T$ and fixed **chemical potential** $\mu$ — the **grand canonical** ensemble
({numref}`fig-gc-reservoir`). Repeating the reservoir argument of [§5.4](microstates-entropy-temperature.ipynb), but now expanding the
reservoir entropy in *two* small quantities (the energy and the number handed over), the
probability of a microstate picks up a factor for each: $P_s\propto e^{-\beta(E_s-\mu N_s)}$
{eq}`eq-grand`. The normalizer is the **grand partition function** $\Xi=\sum_s e^{-\beta(E_s-\mu
N_s)}$, which regroups into a sum over particle number of the canonical $Z_N$ weighted by the
fugacity $z=e^{\beta\mu}$. And just as $F=-kT\ln Z$ was the canonical master potential, $\Phi=-kT
\ln\Xi$ is the grand potential — the very $\Phi(T,V,\mu)$ that [§5.7](potentials-legendre-maxwell.ipynb) built by Legendre transform,
now computed from the microscopics.

**This exercise (worked).** Build the grand-canonical probabilities for a small open system (a
single site that may be empty or occupied) and confirm they are normalized by $\Xi$.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    P_s.sum(), 1.0, "the grand canonical probabilities are normalized by Ξ", rtol=1e-12
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — The open ideal gas: $\Xi=e^{zZ_1}$ and Poisson particle number

The grand canonical ensemble proves its worth immediately for the ideal gas, and it hands us back an old
friend. Because the gas is non-interacting, its $N$-particle partition function is $Z_N=Z_1^N/N!$
(the factorization of [§5.8](partition-function.ipynb), with the Gibbs $1/N!$ of [§5.6](ideal-gas-fundamental-relation.ipynb)), so the grand sum is just the exponential
series, $\Xi=\sum_N(zZ_1)^N/N!=e^{zZ_1}$ {eq}`eq-gc-poisson`. Reading off the term for each $N$, the
probability that the open system contains exactly $N$ particles is $P(N)=(zZ_1)^N e^{-zZ_1}/N!$ — a
**Poisson distribution**, precisely the law we built in [§5.2](probability.ipynb) for rare independent events, now
arrived at as the *equilibrium particle statistics of an open ideal gas* ({numref}`fig-gc-poisson`).
Its mean is $\langle N\rangle=zZ_1$, and its defining signature is that the variance equals the
mean, $\operatorname{Var}(N)=\langle N\rangle$, so the relative fluctuation is $\sigma_N/\langle
N\rangle=1/\sqrt{\langle N\rangle}$ — already the $1/\sqrt N$ that will prove the ensembles
equivalent. The factorial overflows for any real gas, so we compute the pmf in log space with
`scipy.special.gammaln`.

**This exercise (worked).** For an open ideal gas with $zZ_1=20$, build the Poisson pmf with
`poisson_pmf_logspace`, and confirm $\langle N\rangle=\operatorname{Var}(N)=zZ_1$ and $\sigma_N/
\langle N\rangle=1/\sqrt{\langle N\rangle}$.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    [mean_N, var_N],
    [lam, lam],
    "the open ideal gas has Poisson particle number, ⟨N⟩ = Var(N) = zZ₁",
    rtol=1e-3,
)
validate.close(
    np.sqrt(var_N) / mean_N,
    1 / np.sqrt(mean_N),
    "the relative number fluctuation is σ_N/⟨N⟩ = 1/√⟨N⟩",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Number fluctuations and the compressibility

Now the first new fluctuation–response relation, and the central one of this notebook. We saw the
variance of $N$ equals its mean for the ideal gas; the deeper statement is *why* a variance should
be a response at all. Differentiating $\ln\Xi$ once gives the mean number, $\langle N\rangle=
\tfrac1\beta\partial\ln\Xi/\partial\mu$; differentiating again gives the variance, so
$\operatorname{Var}(N)=kT(\partial\langle N\rangle/\partial\mu)_{T,V}$ {eq}`eq-fluct-response`. The
right-hand side is a **response function**: how much the particle number changes when we nudge the
chemical potential. A little thermodynamics turns it into the **isothermal compressibility**,
$\operatorname{Var}(N)=\langle N\rangle^2kT\kappa_T/V$ with $\kappa_T=-\tfrac1V(\partial V/\partial
P)_T$. The physical reading is immediate: a nearly **incompressible** system (small $\kappa_T$) has
small number fluctuations — squeeze it and nothing gives — while a system whose compressibility
*diverges* has macroscopic number fluctuations. For the ideal gas both forms collapse to
$\operatorname{Var}(N)=\langle N\rangle$, the Poisson result, which is the check.

**This exercise (worked).** Compute $kT(\partial\langle N\rangle/\partial\mu)$ for the ideal gas by
differentiating $\langle N\rangle(\mu)=e^{\beta\mu}Z_1$ (`numpy.gradient`), and confirm it equals
$\operatorname{Var}(N)=\langle N\rangle$; then confirm the compressibility form $\langle N\rangle^2
kT\kappa_T/V$ gives the same.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    Var_N_response,
    N_here,
    "the particle-number variance equals kT(∂⟨N⟩/∂μ) — the compressibility (fluctuation = response)",
    rtol=1e-3,
)
validate.close(
    Var_N_kappa,
    N_here,
    "the same variance equals ⟨N⟩²kTκ_T/V, with κ_T measured from the isotherm",
    rtol=1e-3,
)

## Exercise 4 — The fluctuation–response principle, collected

We pause to gather what is becoming a pattern into a single principle, because it is one of the
most useful ideas in the subject and deserves to be seen whole. Three times now a **variance** has
turned out to equal a **response function** ({numref}`fig-gc-family`). The canonical energy
fluctuation was the heat capacity, $\operatorname{Var}(E)=kT^2C_V$ ([§5.8](partition-function.ipynb)). The grand-canonical number
fluctuation is the compressibility, $\operatorname{Var}(N)=kT(\partial\langle N\rangle/\partial\mu)
\propto\kappa_T$ (just now). And the magnetization fluctuation is the susceptibility,
$\operatorname{Var}(M)=kT\chi$ — the relation that will diverge at the Ising critical point of [§5.10](ising-emergence-universality.ipynb).
These are one theorem, the **static fluctuation–dissipation theorem**: a susceptibility — the
derivative of an extensive variable with respect to its conjugate intensive field — equals the
variance of that variable, up to a factor of $kT$ ($kT^2$ for the energy–heat-capacity pair).
The reason is always the same, that both the
response and the variance are *second* derivatives of the same free energy. The practical power is
enormous: to measure a response function in a simulation one need not apply a field and measure the
change — one simply lets the system sit at equilibrium and watches it fluctuate. This is exactly
how $C_V$ came out of the Metropolis energy fluctuations in [§5.8](partition-function.ipynb), and how $\kappa_T$ and $\chi$ are
obtained in practice.

**This exercise (worked).** Confirm the energy–heat-capacity instance numerically for the two-state
system — $\operatorname{Var}(E)$ obtained as the second $\beta$-derivative of $\ln Z$ by a central
finite difference, graded against the closed-form $C_V$ through $\operatorname{Var}(E)/kT^2=C_V$ —
to stand beside the number–compressibility instance of Exercise 3: two members of the one family.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    Var_E / T4**2,
    C_V,
    "Var(E) measured as ∂²lnZ/∂β² equals kT²C_V — a fluctuation–response relation",
    rtol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — The canonical energy distribution narrows onto the microcanonical value

We turn to the equivalence of ensembles, and the mechanism is the sharpness of fluctuations. A
canonical system does *not* have a definite energy — it trades energy with its bath — so strictly
it is described by a *distribution* of energies, $P(E)\propto\Omega(E)e^{-\beta E}$: the number of
microstates at energy $E$ (rising fast) times the Boltzmann suppression (falling fast). The product
is sharply peaked, and the peak sits exactly where $\partial\ln\Omega/\partial E=\beta$, i.e. where
$\partial S/\partial E=1/T$ — which is the *microcanonical* energy at temperature $T$
{eq}`eq-gc-equivalence`. So the canonical ensemble's most probable energy is the microcanonical energy;
the only question is how tightly it is pinned there. The answer is the $1/\sqrt N$ of [§5.3](large-n-limit.ipynb): the
relative width $\sigma_E/\langle E\rangle$ shrinks as $1/\sqrt N$ ({numref}`fig-gc-narrowing`), from
of order one for a handful of spins to a part in a thousand by $N=10^6$. As $N\to\infty$ the
canonical energy distribution becomes a spike at the microcanonical value — the two ensembles
describe the same state.

**This exercise (worked).** For $N$ two-state spins, build the canonical energy distribution with
`canonical_energy_distribution`, confirm its mean energy per spin equals the microcanonical value
$-\tanh\beta\varepsilon$, and confirm the relative width falls as $1/\sqrt N$ (from $\approx0.60$ at
$N=2$ to $\approx0.019$ at $N=2000$). The animation shows the distribution narrowing as $N$ grows.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    e_canonical_per_spin,
    e_micro_per_spin,
    "the canonical energy distribution peaks at the microcanonical energy −tanh βε",
    rtol=1e-2,
)
validate.close(
    widths[2000] * np.sqrt(2000),
    widths[2] * np.sqrt(2),
    "the canonical energy fluctuation vanishes as 1/√N (the relative width scales as N^(−1/2))",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — The grand-canonical particle number narrows onto the canonical value

The same collapse happens on the particle-number side, and it completes the chain. The grand
canonical ensemble lets $N$ fluctuate, with the Poisson distribution of Exercise 2; but that
distribution's relative width is $\sigma_N/\langle N\rangle=1/\sqrt{\langle N\rangle}$
{eq}`eq-gc-equivalence`, which vanishes as the system grows. So a macroscopic open system has, for all
thermodynamic purposes, a *fixed* particle number equal to $\langle N\rangle$ — the canonical value
({numref}`fig-gc-poisson` already showed the relative narrowing). Put the two collapses together:
the canonical energy distribution pins onto the microcanonical energy (Exercise 5), and the
grand-canonical number distribution pins onto the canonical number (here). The three ensembles,
which differ only in what they let fluctuate, become indistinguishable once the fluctuations are
negligible — which, by [§5.3](large-n-limit.ipynb), they are at macroscopic $N$.

**This exercise (worked).** Confirm the grand-canonical relative number fluctuation is exactly
$1/\sqrt{\langle N\rangle}$ across several mean occupations, so it $\to0$ in the thermodynamic
limit — the grand canonical ensemble effectively fixing $N$ at the canonical value.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    [rel_N[5.0], rel_N[5000.0]],
    [1 / np.sqrt(5.0), 1 / np.sqrt(5000.0)],
    "the grand-canonical number collapses onto the canonical value as σ_N/⟨N⟩ = 1/√⟨N⟩",
    rtol=1e-3,
)

## Exercise 7 — One quantity, three ensembles, one answer

Here is the payoff the whole notebook has been driving at. If the ensembles are equivalent, then a
thermodynamic quantity computed in any of them must give the same number — and we can simply check
it. We take the energy per particle of the ideal gas and compute it three ways {eq}`eq-gc-equivalence`:
**microcanonically**, from $1/T=\partial S/\partial E$, which gives $E=\tfrac32NkT$ and so $\tfrac32
kT$ per particle (the equipartition of [§5.6](ideal-gas-fundamental-relation.ipynb)); **canonically**, as $-\partial\ln Z_1/\partial\beta=
\tfrac32kT$; and **grand-canonically**, as $\langle E\rangle/\langle N\rangle$, the Poisson-weighted
energy over the Poisson-weighted number, again $\tfrac32kT$. The three agree exactly. What
*distinguishes* the ensembles is only their fluctuations — the canonical energy spread and the
grand-canonical number spread — and those both vanish as $1/\sqrt N$ ({numref}`fig-gc-equivalence`).
So the freedom is real and practical: compute any quantity in whichever ensemble is most convenient
(often the grand canonical, where the unconstrained sums factorize most easily), confident that the
thermodynamic-limit answer is ensemble-independent.

**This exercise (worked).** Compute the ideal-gas energy per particle microcanonically, canonically
(via `numpy.gradient` of $\ln Z_1$ over $\beta$), and grand-canonically — from the grand
potential, $\langle E\rangle/\langle N\rangle=-(\partial Z_1/\partial\beta)/Z_1$, differencing
$Z_1(\beta)$ itself with `numpy.gradient` — and confirm all three equal $\tfrac32kT$; then plot
the distinguishing fluctuations vanishing as $1/\sqrt N$.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    [e_micro, e_canonical, e_grand],
    [1.5 * T7, 1.5 * T7, 1.5 * T7],
    "the three ensembles give the same per-particle energy in the thermodynamic limit",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — Measuring a response function from fluctuations

The fluctuation–response principle is not only an identity to admire; it is a working tool, and
this exercise turns it into one. The claim of Exercise 4 was that a response function *equals* a
fluctuation, so we ought to be able to **measure** a response — say the heat capacity — without ever
changing the temperature and watching the energy respond. Instead we let the system sit at one
temperature, sample its equilibrium energy with the Metropolis algorithm of [§5.8](partition-function.ipynb), measure the
*variance* of that energy, and divide by $kT^2$. For $N$ two-state spins at $\beta=1$ this should
return the exact per-spin heat capacity $(\varepsilon/T)^2\operatorname{sech}^2\beta\varepsilon$.
It does, to sampling accuracy — which is precisely how heat capacities, compressibilities, and
susceptibilities are extracted from Monte Carlo and molecular-dynamics runs in practice (and in
MMM). Response from fluctuation, measured.

**This exercise (student).** Sample $N$ independent two-state spins with a vectorized Metropolis
sweep (`numpy.random.default_rng`, the acceptance `rng.random(N) < np.exp(-beta*dE)`), measure the
energy variance, and recover the per-spin heat capacity $C_V/N=\operatorname{Var}(E)/(NkT^2)$,
comparing to the exact value.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    CV_per_spin_measured,
    CV_per_spin_exact,
    "a response function (the heat capacity) can be measured from equilibrium energy fluctuations",
    rtol=8e-2,
)

## Exercise 9 — Where the equivalence breaks: the critical point

We end at the one place all of this machinery strains, because that is where the interesting
physics begins. Everything in this notebook rested on fluctuations being *small* — vanishing as
$1/\sqrt N$ — which made the ensembles equivalent. But the fluctuation–response relations cut both
ways: if a response function *diverges*, then so do the fluctuations {eq}`eq-criticality`. A
diverging susceptibility, $\chi\to\infty$, means a diverging magnetization variance,
$\operatorname{Var}(M)=kT\chi\to\infty$ — fluctuations that do **not** shrink as $1/\sqrt N$ but
instead grow to span the entire system ({numref}`fig-gc-critical`). At such a **critical point** the
different ensembles can give different answers, and the clean equivalence we proved fails. This is
not a defect of the theory; it is the theory pointing at where the physics is richest. A diverging
susceptibility with system-spanning fluctuations is the fingerprint of a **phase transition** —
and the Ising model of [§5.10](ising-emergence-universality.ipynb) is exactly this story: as the temperature approaches $T_c$ the
susceptibility blows up, fluctuations correlate across the whole magnet, and spontaneous order
appears out of a model with no built-in preference. The whole ensemble apparatus has been building
toward the place where it most dramatically breaks.

**This exercise (worked).** Take a model susceptibility that diverges at a critical temperature,
$\chi\propto1/|T-T_c|$, and confirm that the fluctuation–response relation $\operatorname{Var}(M)=
kT\chi$ makes the magnetization fluctuation diverge in lock-step — the signature of a critical
point, where ensemble equivalence fails.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.check(
    chi[0] > 10 * chi[-1] and Var_M[0] > 10 * Var_M[-1],
    "in the model, response and fluctuation diverge together approaching T_c — where ensemble equivalence fails (the §5.10 preview)",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 10 — One theory, three ensembles, and the role of fluctuations

Look back at the structure now complete. We built three ensembles — microcanonical, canonical, and
grand canonical — distinguished only by what they hold fixed and what they let fluctuate: energy
and number, then just number, then nothing. We proved them **one**, and the proof was
fluctuations: the canonical energy distribution pins onto the microcanonical energy, the
grand-canonical number pins onto the canonical number, each with a relative width that vanishes as
$1/\sqrt N$ ([§5.3](large-n-limit.ipynb)), so in the thermodynamic limit the three describe the same physics and one is
free to choose the most convenient. Along the way the fluctuations turned out to *be* the response
functions — the heat capacity, the compressibility, the susceptibility — the static
fluctuation–dissipation theorem, which is how response is measured in practice. And the one place
the fluctuations do **not** vanish, the critical point, is exactly where the next notebook goes.

**This exercise (synthesis).** No new computation: the architecture is the result. Three ensembles,
proven equivalent by the smallness of fluctuations; fluctuations that are themselves the response
functions; and a single exception — the critical point — where the fluctuations grow without bound.
That exception is the doorway to [§5.10](ising-emergence-universality.ipynb): the Ising model, where a diverging susceptibility and
system-spanning fluctuations produce a genuine phase transition and the spontaneous emergence of
order.

## Notebook summary

This notebook completed the ensemble theory and made fluctuations its spine: the grand canonical
ensemble, the fluctuation–response relations, and the equivalence of ensembles.

- **The grand canonical ensemble** {eq}`eq-grand`: a system open to energy *and* particle exchange
  at fixed $(T,\mu)$ has $P_s=e^{-\beta(E_s-\mu N_s)}/\Xi$, with the grand potential $\Phi=-kT\ln\Xi$
  of [§5.7](potentials-legendre-maxwell.ipynb) grounded statistically.
- **The open ideal gas** {eq}`eq-gc-poisson`: $\Xi=e^{zZ_1}$ and the particle number is **Poisson**
  (the [§5.2](probability.ipynb) payoff), $\langle N\rangle=\operatorname{Var}(N)=zZ_1$, $\sigma_N/\langle N\rangle=
  1/\sqrt{\langle N\rangle}$, computed in log space via `gammaln`.
- **Fluctuation–response** {eq}`eq-fluct-response`: $\operatorname{Var}(E)=kT^2C_V$ ([§5.8](partition-function.ipynb)),
  $\operatorname{Var}(N)=kT\partial_\mu\langle N\rangle\propto\kappa_T$ (here, both forms verified),
  $\operatorname{Var}(M)=kT\chi$ — one theorem, response **is** fluctuation, and the way simulations
  measure response functions.
- **Equivalence of ensembles** {eq}`eq-gc-equivalence`: the canonical energy distribution peaks at the
  microcanonical energy with width $\propto1/\sqrt N$ ($0.60\to0.019$ for $N=2\to2000$), and the
  grand-canonical Poisson number collapses as $1/\sqrt{\langle N\rangle}$; one quantity computed
  three ways agrees ($\tfrac32kT$ per particle).
- **The critical point** {eq}`eq-criticality`: a diverging response is a diverging fluctuation that
  does *not* vanish as $1/\sqrt N$ — where equivalence breaks and the physics gets interesting.

Three ensembles, proven one by the smallness of fluctuations; fluctuations that are the response
functions; and one exception — the critical point — that is the doorway to the volume's capstone.

## Outlook

- **The Ising model ([§5.10](ising-emergence-universality.ipynb)).** The critical point where the susceptibility diverges, fluctuations
  span the system, ensemble equivalence strains, and spontaneous order emerges — the volume's
  capstone, computed with the Metropolis algorithm of [§5.8](partition-function.ipynb).
- **The fluctuation–dissipation theorem, in full.** Its dynamical form relates equilibrium
  fluctuations to the response to *time-dependent* perturbations — a pointer beyond this volume.
- **Open and reacting systems.** The grand canonical ensemble is the natural language for
  adsorption, chemical equilibrium, and phase coexistence — the chemical potential of [§5.6](ideal-gas-fundamental-relation.ipynb) in action.
- **Quantum statistics (Volume VII).** The same $\Xi$ with quantum $Z_N$ gives Fermi–Dirac and
  Bose–Einstein statistics in the occupation-number formalism — once Volume VI builds the quantum
  mechanics.
- **Cross-reference** [§5.2](probability.ipynb) (Poisson), [§5.3](large-n-limit.ipynb) ($1/\sqrt N$, large-$N$), [§5.7](potentials-legendre-maxwell.ipynb) ($\Phi$, response functions),
  [§5.8](partition-function.ipynb) (energy fluctuations, $\operatorname{Var}(E)=kT^2C_V$).

In [ ]:
from ecp.style import footer

footer()